In [ ]:
import sys
!{sys.executable} -m pip install torch torch_geometric yfinance pandas numpy scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.7 MB/s eta 0:00:00


In [ ]:
def backtest_buy_and_hold(
    df: pd.DataFrame,
    tickers: list,
    lookback: int = LOOKBACK,
    rebalance_freq: int = REBALANCE_CANDLES,
) -> pd.DataFrame:
    """
    Equal-weight buy & hold baseline over the exact same test period
    and step indices used by backtest().

    No rebalancing — shares are fixed at entry and drift freely.
    Comparable to backtest() output: same dates, same periods_per_year.
    """
    n_bars = len(df)
    steps  = list(range(lookback + 4, n_bars - rebalance_freq, rebalance_freq))

    assert len(steps) > 0, "Test set too small — no valid steps."

    n          = len(tickers)
    init_idx   = steps[0]
    init_price = df["Close"][tickers].iloc[init_idx].values.astype(np.float64)

    # Equal-weight: allocate 1/N capital to each stock at entry
    # shares[i] = (1/N) / price[i]  — normalised so total portfolio = 1.0
    shares = (1.0 / n) / init_price         # (n,)

    periods_per_year = {
        CandleFreq.DAILY:   252 / rebalance_freq,
        CandleFreq.WEEKLY:  52  / rebalance_freq,
        CandleFreq.MONTHLY: 12  / rebalance_freq,
    }[CANDLE_FREQ]

    records = []
    prev_port_val = 1.0

    for idx in steps:
        prices    = df["Close"][tickers].iloc[idx].values.astype(np.float64)
        port_val  = float((shares * prices).sum())

        # Current drifted weights (informational)
        position_vals = shares * prices
        weights       = position_vals / position_vals.sum()

        # Return since last step
        fut_idx   = idx + rebalance_freq
        fut_price = df["Close"][tickers].iloc[fut_idx].values.astype(np.float64)
        step_ret  = float((shares * fut_price).sum() / (shares * prices).sum() - 1)

        records.append({
            "date":         df.index[idx],
            "portfolio_ret": step_ret,
            **{t: float(w) for t, w in zip(tickers, weights)},
        })

    result = pd.DataFrame(records).set_index("date")
    result["cumulative_ret"] = (1 + result["portfolio_ret"]).cumprod() - 1

    r   = result["portfolio_ret"]
    ann = (1 + r.mean()) ** periods_per_year - 1
    sh  = r.mean() / (r.std() + 1e-8) * math.sqrt(periods_per_year)
    pv  = result["cumulative_ret"] + 1
    mdd = ((pv.cummax() - pv) / pv.cummax()).max()

    print(f"\nBuy & Hold Baseline  [equal-weight 1/{n}, no rebalancing]")
    print(f"  Entry date        : {result.index[0].date()}")
    print(f"  Exit date         : {result.index[-1].date()}")
    print(f"  Total return      : {result['cumulative_ret'].iloc[-1]*100:.2f}%")
    print(f"  Annualised return : {ann*100:.2f}%")
    print(f"  Sharpe            : {sh:.3f}")
    print(f"  Max drawdown      : {mdd*100:.2f}%")

    # Show weight drift from equal to final
    print(f"\n  Weight drift (entry -> exit):")
    final_weights = result[[t for t in tickers]].iloc[-1]
    for t in tickers:
        print(f"    {t:<6}  {100/n:.1f}%  ->  {final_weights[t]*100:.1f}%")

    return result


def main_baseline():
    """
    Runs the buy & hold baseline on the same test split used by the model.
    Call this after main() to get a direct comparison.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"

    end_date   = datetime.today().strftime("%Y-%m-%d")
    start_date = (datetime.today() - timedelta(days=HISTORY_YEARS * 365)).strftime("%Y-%m-%d")

    df = fetch_price_data(TICKERS, start_date, end_date, freq=CANDLE_FREQ)
    df = df.ffill().dropna()

    split   = int(len(df) * 0.75)
    test_df = df.iloc[split:]

    print(f"Test set: {len(test_df)} bars | "
          f"{test_df.index[0].date()} -> {test_df.index[-1].date()}")

    bh_result = backtest_buy_and_hold(test_df, TICKERS)

    return bh_result


if __name__ == "__main__":
    # Run model
    model, model_result, weights = main()

    print("\n" + "=" * 60)

    # Run baseline on identical test window
    bh_result = main_baseline()

    # Side-by-side comparison
    print("\n" + "=" * 60)
    print("COMPARISON SUMMARY")
    print("=" * 60)

    def summarise(result, label):
        r   = result["portfolio_ret"]
        ppy = {
            CandleFreq.DAILY:   252 / REBALANCE_CANDLES,
            CandleFreq.WEEKLY:  52  / REBALANCE_CANDLES,
            CandleFreq.MONTHLY: 12  / REBALANCE_CANDLES,
        }[CANDLE_FREQ]
        ann = (1 + r.mean()) ** ppy - 1
        sh  = r.mean() / (r.std() + 1e-8) * math.sqrt(ppy)
        pv  = result["cumulative_ret"] + 1
        mdd = ((pv.cummax() - pv) / pv.cummax()).max()
        tot = result["cumulative_ret"].iloc[-1]
        print(f"  {label:<25} | total: {tot*100:+6.1f}% | "
              f"ann: {ann*100:+5.1f}% | sharpe: {sh:.3f} | mdd: {mdd*100:.1f}%")

    summarise(model_result, "GAT + GRU model")
    summarise(bh_result,    "Buy & Hold (1/N)")

Device      : cuda
Candle freq : WEEKLY  (lookback=52, rebalance_every=1 candle(s))
Data range  : 2016-03-22 -> 2026-03-20
Candle freq : WEEKLY | bars: 522 | lookback: 52 candles | rebalance every: 1 candle(s)
Bars after resample & dropna: 522
Model parameters: 86,019


Training | WEEKLY candles | 334 rebalance steps x 100 epochs
Stocks : ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'JPM', 'V', 'UNH']
────────────────────────────────────────────────────────────
Epoch   1/100 | avg loss: -0.2229 | lr: 3.00e-04 | updates: 27
Epoch   5/100 | avg loss: -0.2609 | lr: 2.98e-04 | updates: 27
Epoch  10/100 | avg loss: -0.2811 | lr: 2.93e-04 | updates: 27
Epoch  15/100 | avg loss: -0.3053 | lr: 2.84e-04 | updates: 27
Epoch  20/100 | avg loss: -0.3148 | lr: 2.71e-04 | updates: 27
Epoch  25/100 | avg loss: -0.3177 | lr: 2.56e-04 | updates: 27
Epoch  30/100 | avg loss: -0.3333 | lr: 2.38e-04 | updates: 27
Epoch  35/100 | avg loss: -0.3437 | lr: 2.18e-04 | updates: 27
Epoch  40/100 | a

In [ ]:
def save_model(model, path: str = "gat_portfolio.pth"):
    torch.save({
        "model_state_dict":  model.state_dict(),
        "n_stocks":          model.n_stocks,
        "candle_freq":       CANDLE_FREQ.name,
        "lookback":          LOOKBACK,
        "tickers":           TICKERS,
        "d_model":           D_MODEL,
        "n_layers_tf":       N_LAYERS_TRANSFORMER,
        "n_heads_gat":       N_HEADS_GAT,
        "gat_layers":        GAT_LAYERS,
        "dropout":           DROPOUT,
        "features_per_bar":  FEATURES_PER_BAR,
        "saved_at":          datetime.today().strftime("%Y-%m-%d %H:%M:%S"),
    }, path)
    print(f"Model saved to {path}")


def load_model(path: str = "gat_portfolio.pth", device: str = "cpu") -> GATPortfolioNet:
    checkpoint = torch.load(path, map_location=device)
    model = GATPortfolioNet(
        n_stocks    = checkpoint["n_stocks"],
        n_features  = checkpoint["features_per_bar"],
        d_model     = checkpoint["d_model"],
        n_layers_tf = checkpoint["n_layers_tf"],
        n_heads_gat = checkpoint["n_heads_gat"],
        gat_layers  = checkpoint["gat_layers"],
        dropout     = checkpoint["dropout"],
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()
    print(f"Model loaded from {path}")
    print(f"  Saved at    : {checkpoint['saved_at']}")
    print(f"  Tickers     : {checkpoint['tickers']}")
    print(f"  Candle freq : {checkpoint['candle_freq']}")
    print(f"  Lookback    : {checkpoint['lookback']}")
    return model


# --- Call in main() after training, before backtest: ---
#model = train(model, train_df, TICKERS, device=device)
save_model(model, path="gat_portfolio.pth")

# --- To reload later: ---
# model = load_model("gat_portfolio.pth", device=device)

Model saved to gat_portfolio.pth


In [ ]:
def model_simulation(
    model_path: str = "gat_portfolio.pth",
    initial_capital: float = 100_000.0,
    device: str = "cpu",
) -> pd.DataFrame:
    """
    Loads the saved model and simulates weekly rebalancing through 2017.
    Starts with equal-weight allocation and adjusts every week per model output.
    Tracks capital, weights, turnover, and transaction costs.
    """
    # ── Load model ────────────────────────────────────────────────────────
    model      = load_model(model_path, device=device)
    checkpoint = torch.load(model_path, map_location=device)
    tickers    = checkpoint["tickers"]
    lookback   = checkpoint["lookback"]
    n          = len(tickers)

    # ── Fetch data ────────────────────────────────────────────────────────
    # Need data before 2017 to fill the lookback window (52 weekly candles)
    # 52 weeks lookback + buffer → start from early 2015
    df = fetch_price_data(
        tickers,
        start = "2015-01-01",
        end   = "2017-12-31",
        freq  = CandleFreq[checkpoint["candle_freq"]],
    )
    df = df.ffill().dropna()

    # Isolate the 2017 weekly bars
    sim_df    = df[df.index.year == 2017]
    sim_dates = sim_df.index.tolist()

    assert len(sim_dates) >= 2, "No 2017 weekly bars found."
    print(f"\n2017 Simulation | {len(sim_dates)} weekly bars | "
          f"{sim_dates[0].date()} -> {sim_dates[-1].date()}")
    print(f"Tickers  : {tickers}")
    print(f"Capital  : ${initial_capital:,.0f}")
    print(f"{'─'*70}")

    # ── State ─────────────────────────────────────────────────────────────
    history       = []              # regret memory — empty at start of year
    capital       = initial_capital
    prev_weights  = np.ones(n) / n  # start equal-weight
    records       = []

    for i, date in enumerate(sim_dates[:-1]):   # last bar has no next week
        # Global index of this date in the full df (needed for feature tensor)
        global_idx = df.index.get_loc(date)

        # Need at least lookback + 4 bars before this point
        if global_idx < lookback + 4:
            print(f"  [{date.date()}] Skipping — not enough lookback history yet.")
            continue

        # ── Build inputs ──────────────────────────────────────────────────
        x      = build_feature_tensor(df, tickers, global_idx, lookback).to(device)
        ei, ea = build_correlation_graph(df, tickers, global_idx, lookback)
        ei, ea = ei.to(device), ea.to(device)

        # ── Model inference ───────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            weights_tensor = model(x, ei, ea, history=history, device=device)

        if torch.isnan(weights_tensor).any():
            print(f"  [{date.date()}] NaN weights — holding previous allocation.")
            new_weights = prev_weights
        else:
            new_weights = weights_tensor.cpu().numpy()

        # ── Turnover & transaction cost (5bps per unit traded) ────────────
        turnover      = float(np.abs(new_weights - prev_weights).sum())
        tx_cost_rate  = 0.0005              # 5 basis points per side
        tx_cost       = capital * turnover * tx_cost_rate

        # ── Realised return (this week → next week) ───────────────────────
        next_date  = sim_dates[i + 1]
        cur_price  = df["Close"][tickers].loc[date].values.astype(np.float64)
        next_price = df["Close"][tickers].loc[next_date].values.astype(np.float64)
        ret_vec    = (next_price - cur_price) / (cur_price + 1e-8)
        port_ret   = float((new_weights * ret_vec).sum())

        # ── Update capital ────────────────────────────────────────────────
        capital = (capital - tx_cost) * (1 + port_ret)

        # ── Update regret history ─────────────────────────────────────────
        history.append((
            torch.tensor(new_weights, dtype=torch.float32),
            torch.tensor(ret_vec,     dtype=torch.float32),
        ))
        if len(history) > model.memory.memory_len:
            history.pop(0)

        records.append({
            "date":           date,
            "capital":        capital,
            "weekly_ret":     port_ret,
            "turnover":       turnover,
            "tx_cost":        tx_cost,
            **{t: float(w) for t, w in zip(tickers, new_weights)},
        })

        # ── Weekly print ──────────────────────────────────────────────────
        top2 = sorted(zip(tickers, new_weights), key=lambda x: -x[1])[:2]
        top2_str = "  ".join(f"{t}={w*100:.1f}%" for t, w in top2)
        print(f"  [{date.date()}]  ret={port_ret*100:+5.2f}%  "
              f"capital=${capital:>10,.0f}  "
              f"top2: {top2_str}  "
              f"turnover={turnover*100:.1f}%")

        prev_weights = new_weights

    # ── Build results ──────────────────────────────────────────────────────
    result = pd.DataFrame(records).set_index("date")
    result["cumulative_ret"] = (1 + result["weekly_ret"]).cumprod() - 1

    r   = result["weekly_ret"]
    ann = (1 + r.mean()) ** 52 - 1
    sh  = r.mean() / (r.std() + 1e-8) * math.sqrt(52)
    pv  = result["cumulative_ret"] + 1
    mdd = ((pv.cummax() - pv) / pv.cummax()).max()

    total_tx = result["tx_cost"].sum()
    avg_turn = result["turnover"].mean()

    print(f"\n{'─'*70}")
    print(f"2017 Simulation Results")
    print(f"  Start capital     : ${initial_capital:>12,.2f}")
    print(f"  End capital       : ${capital:>12,.2f}")
    print(f"  Total return      : {result['cumulative_ret'].iloc[-1]*100:+.2f}%")
    print(f"  Annualised return : {ann*100:+.2f}%")
    print(f"  Sharpe            : {sh:.3f}")
    print(f"  Max drawdown      : {mdd*100:.2f}%")
    print(f"  Total tx costs    : ${total_tx:,.2f}")
    print(f"  Avg weekly turnover: {avg_turn*100:.1f}%")

    return result
def buy_and_hold(
    initial_capital: float = 100_000.0,
) -> pd.DataFrame:
    """
    Equal-weight buy & hold through 2017 — no rebalancing, no transaction costs.
    Uses identical dates and price data as run_2017_simulation() for direct comparison.
    """
    tickers = TICKERS
    n       = len(tickers)

    # ── Fetch same data as simulation ─────────────────────────────────────
    df = fetch_price_data(
        tickers,
        start = "2015-01-01",
        end   = "2017-12-31",
        freq  = CANDLE_FREQ,
    )
    df = df.ffill().dropna()

    sim_df    = df[df.index.year == 2017]
    sim_dates = sim_df.index.tolist()

    assert len(sim_dates) >= 2, "No 2017 weekly bars found."
    print(f"\n2017 Buy & Hold | {len(sim_dates)} weekly bars | "
          f"{sim_dates[0].date()} -> {sim_dates[-1].date()}")
    print(f"Tickers  : {tickers}")
    print(f"Capital  : ${initial_capital:,.0f}")
    print(f"{'─'*70}")

    # ── Entry: equal weight on first 2017 bar ─────────────────────────────
    entry_price = df["Close"][tickers].loc[sim_dates[0]].values.astype(np.float64)
    shares      = (initial_capital / n) / entry_price   # fixed for entire year

    capital      = initial_capital
    records      = []

    for i, date in enumerate(sim_dates[:-1]):
        next_date  = sim_dates[i + 1]
        cur_price  = df["Close"][tickers].loc[date].values.astype(np.float64)
        next_price = df["Close"][tickers].loc[next_date].values.astype(np.float64)

        # Portfolio value at this bar
        position_vals = shares * cur_price
        port_val      = position_vals.sum()

        # Drifted weights (informational — these change as prices diverge)
        weights   = position_vals / port_val

        # Return this week
        ret_vec   = (next_price - cur_price) / (cur_price + 1e-8)
        port_ret  = float((weights * ret_vec).sum())

        # Update capital
        capital   = port_val * (1 + port_ret)

        records.append({
            "date":        date,
            "capital":     capital,
            "weekly_ret":  port_ret,
            "turnover":    0.0,      # no rebalancing ever
            "tx_cost":     0.0,
            **{t: float(w) for t, w in zip(tickers, weights)},
        })

        # ── Weekly print ──────────────────────────────────────────────────
        top2 = sorted(zip(tickers, weights), key=lambda x: -x[1])[:2]
        top2_str = "  ".join(f"{t}={w*100:.1f}%" for t, w in top2)
        print(f"  [{date.date()}]  ret={port_ret*100:+5.2f}%  "
              f"capital=${capital:>10,.0f}  "
              f"top2: {top2_str}")

    # ── Build results ──────────────────────────────────────────────────────
    result = pd.DataFrame(records).set_index("date")
    result["cumulative_ret"] = (1 + result["weekly_ret"]).cumprod() - 1

    r   = result["weekly_ret"]
    ann = (1 + r.mean()) ** 52 - 1
    sh  = r.mean() / (r.std() + 1e-8) * math.sqrt(52)
    pv  = result["cumulative_ret"] + 1
    mdd = ((pv.cummax() - pv) / pv.cummax()).max()

    # Final drifted weights
    final_price  = df["Close"][tickers].loc[sim_dates[-1]].values.astype(np.float64)
    final_vals   = shares * final_price
    final_w      = final_vals / final_vals.sum()

    print(f"\n{'─'*70}")
    print(f"2017 Buy & Hold Results")
    print(f"  Start capital     : ${initial_capital:>12,.2f}")
    print(f"  End capital       : ${capital:>12,.2f}")
    print(f"  Total return      : {result['cumulative_ret'].iloc[-1]*100:+.2f}%")
    print(f"  Annualised return : {ann*100:+.2f}%")
    print(f"  Sharpe            : {sh:.3f}")
    print(f"  Max drawdown      : {mdd*100:.2f}%")
    print(f"\n  Weight drift (entry -> exit):")
    for t in tickers:
        print(f"    {t:<6}  {100/n:.1f}%  ->  {final_w[tickers.index(t)]*100:.1f}%")

    return result


# ── Call both and compare ─────────────────────────────────────────────────
if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"

    sim_result = model_simulation(
        model_path      = "gat_portfolio.pth",
        initial_capital = 100_000.0,
        device          = device,
    )

    bh_result = buy_and_hold(
        initial_capital = 100_000.0,
    )

    # ── Side-by-side ──────────────────────────────────────────────────────
    print(f"\n{'='*70}")
    print(f"{'FINAL COMPARISON':^70}")
    print(f"{'='*70}")
    print(f"  {'Metric':<22}  {'GAT Model':>14}  {'Buy & Hold':>14}")
    print(f"  {'─'*22}  {'─'*14}  {'─'*14}")

    def stats(r):
        ann = (1 + r["weekly_ret"].mean()) ** 52 - 1
        sh  = r["weekly_ret"].mean() / (r["weekly_ret"].std() + 1e-8) * math.sqrt(52)
        pv  = r["cumulative_ret"] + 1
        mdd = ((pv.cummax() - pv) / pv.cummax()).max()
        tot = r["cumulative_ret"].iloc[-1]
        return tot, ann, sh, mdd

    m_tot, m_ann, m_sh, m_mdd = stats(sim_result)
    b_tot, b_ann, b_sh, b_mdd = stats(bh_result)

    print(f"  {'Total return':<22}  {m_tot*100:>+13.2f}%  {b_tot*100:>+13.2f}%")
    print(f"  {'Annualised return':<22}  {m_ann*100:>+13.2f}%  {b_ann*100:>+13.2f}%")
    print(f"  {'Sharpe':<22}  {m_sh:>14.3f}  {b_sh:>14.3f}")
    print(f"  {'Max drawdown':<22}  {m_mdd*100:>13.2f}%  {b_mdd*100:>13.2f}%")
    print(f"  {'End capital':<22}  ${sim_result['capital'].iloc[-1]:>13,.2f}  "
          f"${bh_result['capital'].iloc[-1]:>13,.2f}")
    print(f"  {'Total tx costs':<22}  ${sim_result['tx_cost'].sum():>13,.2f}  "
          f"${'0.00':>13}")

Model loaded from gat_portfolio.pth
  Saved at    : 2026-03-20 15:08:05
  Tickers     : ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'JPM', 'V', 'UNH']
  Candle freq : WEEKLY
  Lookback    : 52
Candle freq : WEEKLY | bars: 157 | lookback: 52 candles | rebalance every: 1 candle(s)

2017 Simulation | 52 weekly bars | 2017-01-06 -> 2017-12-29
Tickers  : ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'JPM', 'V', 'UNH']
Capital  : $100,000
──────────────────────────────────────────────────────────────────────
  [2017-01-06]  ret=+1.07%  capital=$   101,023  top2: GOOGL=25.2%  TSLA=25.2%  turnover=91.3%
  [2017-01-13]  ret=+0.67%  capital=$   101,705  top2: GOOGL=25.3%  TSLA=25.3%  turnover=0.3%
  [2017-01-20]  ret=+2.93%  capital=$   104,686  top2: GOOGL=25.2%  TSLA=25.2%  turnover=3.0%
  [2017-01-27]  ret=-0.28%  capital=$   104,387  top2: GOOGL=25.2%  TSLA=25.2%  turnover=4.4%
  [2017-02-03]  ret=+2.40%  capital=$   106,885  top2: TSLA=25.0%  V=25.0%  turnover=7.4%